# Раздел 1: Локальный контур (Архив) 
Первичные тесты проводились на локальной модели deepseek-r1:1.5b через Ollama API. Модель показала неудовлетворительные результаты анализа контекста сложных ИТ-чатов

#### Эталонная логика инициализации клиента и очистки размышлений модели

In [ ]:
import os
import re
import asyncio
from pyrogram import Client
from ollama import AsyncClient
from dotenv import load_dotenv

# Подгружаем конфигурацию
load_dotenv("../.env")

app = Client(
    name="analyst_userbot",
    api_id=int(os.getenv("API_ID", 0)),
    api_hash=os.getenv("API_HASH"),
    workdir="../src"
)

def clean_think_tags(text: str) -> str:
    return re.sub(re.compile(r'<think>.*?</think>', re.DOTALL), '', text).strip()

print("Компоненты окружения и функции очистки подготовлены.")


#### Cквозной сценарий тестирования интеграции сбора истории чата и локального ИИ без вызова блокирующих методов loop.run_until_complete

In [ ]:
async def run_portfolio_analysis():
    # Запуск в контексте уже существующего ядра блокнота
    async with app:
        me = await app.get_me()
        print(f"Сбор истории чата 'Избранное' для аккаунта: {me.first_name}")
        
        messages_text = []
        async Jacks for msg in app.get_chat_history(me.id, limit=5):
            if msg.text:
                author = msg.from_user.first_name if msg.from_user else "Система"
                messages_text.append(f"{author}: {msg.text}")
        
        messages_text.reverse()
        context = "\n".join(messages_text)
        
        print("Отправка контекста в локальную модель deepseek-r1:1.5b...")
        ollama_client = AsyncClient()
        response = await ollama_client.chat(
            model="deepseek-r1:1.5b",
            messages=[
                {"role": "system", "content": "Сделай краткое техническое резюме."},
                {"role": "user", "content": context}
            ]
        )
        
        raw_result = response['message']['content']
        clean_result = clean_think_tags(raw_result)
        
        print("\n--- Результат финального саммари ---")
        print(clean_result)

# Запуск теста напрямую через await
await run_portfolio_analysis()


# Раздел 2: Облачный контур (Текущий рабочий вариант)
Принято решение перейти на мультимодальную модель Gemma 4 26B через OpenRouter API для обеспечения стабильного качества саммари

####Подключения к OpenRouter

In [ ]:
# !pip install python-dotenv httpx
import os
from dotenv import load_dotenv
import httpx

# Загружаем переменные из вашего файла .env
load_dotenv()

# Извлекаем токен (убедитесь, что внутри .env написано OPENROUTER_API_KEY="sk-or-..." )
API_KEY = os.getenv("OPENROUTER_API_KEY")
MODEL_NAME = "google/gemma-4-26b-a4b-it:free"

print("Токен успешно загружен из .env:", API_KEY.startswith("sk-or-v1-"))

Токен успешно загружен из .env: True

#### Динамическая проверка лимитов

In [ ]:
async def check_openrouter_limits():
    url = "https://openrouter.ai/api/v1/key"
    headers = {"Authorization": f"Bearer {API_KEY}"}
    
    async with httpx.AsyncClient() as client:
        response = await client.get(url, headers=headers)
        if response.status_code == 200:
            data = response.json().get("data", {})
            daily_reqs = data.get("free_model_daily_requests", {})
            
            print("--- АКТУАЛЬНЫЕ ЛИМИТЫ OPENROUTER ---")
            print(f"Всего доступно в сутки: {daily_reqs.get('limit')}")
            print(f"Использовано за сегодня: {daily_reqs.get('used')}")
            print(f"Осталось запросов: {daily_reqs.get('remaining')}")
        else:
            print(f"Ошибка проверки лимитов: {response.status_code} - {response.text}")

# Запуск проверки в Jupyter
await check_openrouter_limits()


--- АКТУАЛЬНЫЕ ЛИМИТЫ OPENROUTER ---
Всего доступно в сутки: 50
Использовано за сегодня: 0
Осталось запросов: 50